# Data Science — First Principles Reference

**AIcademy Knowledgebase | Data Scientist Track**

**Sections:**
- [DS-1: What is Data?](#ds-1)
- [DS-2: Descriptive Statistics](#ds-2)
- [DS-3: Pandas Core Patterns](#ds-3)
- [DS-4: Visualization](#ds-4)
- [DS-5: What is Machine Learning?](#ds-5)
- [DS-6: Model Evaluation](#ds-6)

---
## DS-1: What is Data? <a id='ds-1'></a>

### The mental model

Data is **recorded observations about the world**. Every dataset has two dimensions:
- **Rows** — individual observations (one person, one transaction, one event)
- **Columns** — features or attributes of each observation (age, salary, country)

### Types of data

| Type | What it is | Examples |
|---|---|---|
| **Numerical (continuous)** | Any number on a scale | Age, salary, temperature |
| **Numerical (discrete)** | Whole numbers only | Number of children, count of orders |
| **Categorical (nominal)** | Named groups, no order | Country, job title, color |
| **Categorical (ordinal)** | Named groups WITH order | Education level, star rating |
| **Boolean** | True/False | Is customer active?, Has disease? |
| **Text** | Free-form strings | Reviews, comments, descriptions |
| **Datetime** | Points in time | Purchase date, login timestamp |

In [ ]:
import pandas as pd
import numpy as np

# Load the AIcademy dataset
df = pd.read_csv("../02_datasets/extended_income_job_country_100_rows.csv")

# Always start by understanding the shape and structure
print(df.shape)       # (rows, columns)
print(df.dtypes)      # data type of each column
print(df.head())      # first 5 rows
print(df.info())      # non-null counts + dtypes
print(df.describe())  # summary stats for numeric columns

---
## DS-2: Descriptive Statistics <a id='ds-2'></a>

### The core measures

**Central tendency** — where is the center of the data?
- **Mean** — average. Sensitive to outliers. One billionaire in the room raises everyone's average salary.
- **Median** — middle value when sorted. Resistant to outliers. Better for skewed data (income, house prices).
- **Mode** — most frequent value. Useful for categorical data.

**Spread** — how spread out is the data?
- **Range** — max minus min. Simple but affected by outliers.
- **Variance** — average squared distance from the mean.
- **Standard deviation (std)** — square root of variance. Same units as the data. Most useful.
- **IQR** — interquartile range (Q3 - Q1). Spread of the middle 50% of data.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../02_datasets/extended_income_job_country_100_rows.csv")
income = df["Average Annual Income (USD)"]

print(f"Mean:   {income.mean():.2f}")
print(f"Median: {income.median():.2f}")
print(f"Std:    {income.std():.2f}")
print(f"Min:    {income.min():.2f}")
print(f"Max:    {income.max():.2f}")
print(f"Q1:     {income.quantile(0.25):.2f}")
print(f"Q3:     {income.quantile(0.75):.2f}")
print(f"IQR:    {income.quantile(0.75) - income.quantile(0.25):.2f}")

---
## DS-3: Pandas Core Patterns <a id='ds-3'></a>

### 3.1 Filtering

In [ ]:
df = pd.read_csv("../02_datasets/extended_income_job_country_100_rows.csv")

# Filter rows by condition
high_earners = df[df["Average Annual Income (USD)"] > 100000]

# Multiple conditions — use & (and) and | (or) with parentheses
tech_high = df[(df["Industry"] == "Technology") & (df["Average Annual Income (USD)"] > 80000)]

# Filter using isin — match multiple values
target_countries = df[df["Country"].isin(["USA", "Canada", "UK"])]

# Filter using str.contains — partial string match
engineers = df[df["Job Title"].str.contains("Engineer", case=False)]

### 3.2 Groupby and Aggregation

In [ ]:
# Average income by industry
industry_avg = df.groupby("Industry")["Average Annual Income (USD)"].mean().sort_values(ascending=False)
print(industry_avg)

# Multiple aggregations at once
summary = df.groupby("Industry")["Average Annual Income (USD)"].agg(["mean", "median", "count", "std"])
print(summary)

# Group by multiple columns
country_industry = df.groupby(["Country", "Industry"])["Average Annual Income (USD)"].mean()
print(country_industry)

### 3.3 Missing Values

In [ ]:
# Check for missing values
print(df.isna().sum())          # count per column
print(df.isna().sum() / len(df) * 100)  # as percentage

# Handle missing values
df["Income"].fillna(df["Income"].median(), inplace=True)  # fill with median
df.dropna(subset=["Job Title"], inplace=True)              # drop rows missing job title
df["Country"].fillna("Unknown", inplace=True)             # fill with placeholder

---
## DS-4: Visualization <a id='ds-4'></a>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../02_datasets/extended_income_job_country_100_rows.csv")

# Distribution of income — histogram
plt.figure(figsize=(10, 5))
plt.hist(df["Average Annual Income (USD)"], bins=20, edgecolor="black")
plt.title("Income Distribution")
plt.xlabel("Annual Income (USD)")
plt.ylabel("Count")
plt.show()

# Average income by industry — bar chart
industry_avg = df.groupby("Industry")["Average Annual Income (USD)"].mean().sort_values()
plt.figure(figsize=(10, 6))
industry_avg.plot(kind="barh")
plt.title("Average Income by Industry")
plt.xlabel("Average Annual Income (USD)")
plt.tight_layout()
plt.show()

# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number])
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

---
## DS-5: What is Machine Learning? <a id='ds-5'></a>

### The mental model

Traditional programming: you write rules → the computer applies them → output.

Machine learning: you give examples (input + correct output) → the computer learns the rules → it applies them to new input.

### Types of ML

| Type | What it does | When to use | Example |
|---|---|---|---|
| **Supervised** | Learns from labeled examples | You have input + correct output pairs | Predicting house prices, spam detection |
| **Unsupervised** | Finds patterns with no labels | You only have input, no labels | Customer segmentation, anomaly detection |
| **Reinforcement** | Learns by trial and error with rewards | Agent interacting with environment | Game-playing AI, robotics |

### A simple end-to-end supervised ML example

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np

df = pd.read_csv("../02_datasets/extended_income_job_country_100_rows.csv")

# Feature engineering: encode categorical columns
le = LabelEncoder()
df["Industry_encoded"] = le.fit_transform(df["Industry"])
df["Country_encoded"] = le.fit_transform(df["Country"])

# Define features (X) and target (y)
X = df[["Industry_encoded", "Country_encoded"]]
y = df["Average Annual Income (USD)"]

# Split into train and test sets
# RULE: Never train and evaluate on the same data — that's cheating
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate on the TEST set (data the model has never seen)
y_pred = model.predict(X_test)

print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
# R² of 1.0 = perfect. 0.0 = no better than predicting the mean. Negative = worse.

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: ${rmse:,.0f}")
# Average prediction error in dollars

---
## DS-6: Model Evaluation <a id='ds-6'></a>

### The train/test split — why it matters

If you train and test on the same data, the model might just memorize the answers — not learn the pattern. This is called **overfitting**.

**Overfitting:** The model performs great on training data but poorly on new data. It memorized noise instead of learning signal.

**Underfitting:** The model performs poorly on both training and test data. The model is too simple.

**The goal:** A model that generalizes — performs well on data it has never seen.

### Key metrics

**For regression (predicting a number):**
| Metric | What it measures | When to use |
|---|---|---|
| MSE / RMSE | Average squared/root-squared error | Default for regression |
| MAE | Average absolute error | When outliers shouldn't dominate |
| R² | % of variance explained (0–1) | How good relative to just predicting the mean |

**For classification (predicting a category):**
| Metric | What it measures | When to use |
|---|---|---|
| Accuracy | % of correct predictions | Balanced classes |
| Precision | Of predicted positives, how many are right | When false positives are costly (spam filter) |
| Recall | Of actual positives, how many did we catch | When false negatives are costly (cancer diagnosis) |
| F1 Score | Harmonic mean of precision and recall | Imbalanced classes |